# CodeGen — Group 45

## Step 9 — does our pipeline still help at 7B? Cascade + repair on the large model

**The question.** Step 8 showed a vanilla Qwen2.5-Coder-7B (4-bit) scores 58.3% — better
than every hand-built 1.5B system. The best-of oracle (1.5B cascade ∪ 7B) reached 65.4%,
and crucially the 1.5B cascade still solved **11 problems the vanilla 7B missed**. That
says our inference-time pipeline is not redundant with scale. This notebook tests it
directly: apply the **same** compile-guided cascade and compiler-feedback repair to the
**7B** and see whether they lift it above 58.3%.

**What runs.** Exactly the Step 6/7 recipe, only the model is the 7B:

1. The 7B vanilla completions are the cascade's first tier — reused from Step 8's
   `step8_qwen7b_4bit.jsonl` (that run *is* K=0), so no regeneration.
2. Two more 7B sweeps for the other tiers: nearest-idiom + K=2, and K=4.
3. **Cascade:** K=0 -> idiom+k2 -> k4, first body that compiles.
4. **Repair:** hand whatever still fails to compile back to the 7B with the rustc error
   (Step 7 base-model repair, completion-style), up to two rounds.

Every gate is a compile signal, so the whole thing stays inference-legal. `evaluate_one`
is byte-identical to every prior step, so the number sits on the same axis.

| Same harness | Score |
|---|---|
| Qwen-1.5B cascade + repair (best small system) | 46.2% |
| Qwen-7B-4bit vanilla (Step 8) | 58.3% |
| best-of (1.5B cascade ∪ 7B vanilla) oracle | 65.4% |
| **7B cascade / 7B cascade+repair (this step)** | **measured below** |

House rules apply. Sections 0-5 are the unchanged harness; Sections 7-8 are the Step 6
RAG machinery reused verbatim; only the 7B load and the 7B sweeps/repair are new.

**Cost note.** Two 7B sweeps over 156 problems plus a short repair pass is roughly an
hour of T4 time. Everything streams to Drive and resumes, so a dropped session is safe.

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [2]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.1 (8bab26

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [3]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 80.2 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [4]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [5]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [6]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6a. Dependencies for 4-bit loading

Same as Step 8: **bitsandbytes** (nf4 kernels) and a recent **accelerate** (`device_map`).

In [7]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes", "accelerate", "transformers"], check=True)
import bitsandbytes as bnb
print("bitsandbytes", bnb.__version__, "ready")

bitsandbytes 0.50.0 ready


## 6b. Load the 4-bit 7B

Same acquisition ladder as Step 8, and by now Step 8 has already stashed the **4-bit copy
on Drive** — so this is a fast ~5 GB Drive load, not the 15 GB download again. If you are
running this before Step 8 on a fresh account, it falls back to ModelScope automatically.

In [8]:
import os, shutil, subprocess, sys, json as _json

MODEL_ID  = "Qwen/Qwen2.5-Coder-7B"      # base (completion) sibling of the 1.5B subject model
MARKER    = "_SAVED_OK"
DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-7b-4bit") if DRIVE_ROOT else None
LOCAL_4BIT_DIR  = "/content/qwen25coder-7b-4bit"
SAVE_7B_TO_DRIVE = True

def _modelscope_download():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"], check=True)
    from modelscope import snapshot_download
    return snapshot_download(MODEL_ID)

def _hf_download():
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=1800)
            from huggingface_hub import snapshot_download
            return snapshot_download(MODEL_ID, local_files_only=True)
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 30 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError("All hubs failed for " + MODEL_ID + ". Run Step 8 first to populate the "
                       "Drive 4-bit cache, or upload it to models/qwen25coder-7b-4bit with _SAVED_OK.")

def fetch_model_dir():
    if DRIVE_MODEL_DIR and os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
        if not os.path.exists(os.path.join(LOCAL_4BIT_DIR, MARKER)):
            print("4-bit 7B found on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_4BIT_DIR, dirs_exist_ok=True)
        print("Using the Drive 4-bit copy")
        return LOCAL_4BIT_DIR, True
    try:
        path = _modelscope_download()
        print("Downloaded raw 7B from ModelScope")
        return path, False
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")
    path = _hf_download()
    print("Downloaded raw 7B from the Hugging Face Hub")
    return path, False

model_dir, prequantized = fetch_model_dir()
cfgp = os.path.join(model_dir, "config.json")
if os.path.exists(cfgp):
    prequantized = prequantized or ("quantization_config" in _json.load(open(cfgp)))
print("model files at:", model_dir, "| already 4-bit:", prequantized)

4-bit 7B found on Drive — copying to local disk (one-time per session)...
Using the Drive 4-bit copy
model files at: /content/qwen25coder-7b-4bit | already 4-bit: True


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bf16
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(model_dir)
if prequantized:
    model = AutoModelForCausalLM.from_pretrained(model_dir, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_dir, quantization_config=BNB, device_map="auto")
model.eval()
GEN_DEVICE = "cuda"
print("7B loaded 4-bit;", f"{model.get_memory_footprint()/1e9:.1f} GB on GPU")

if (not prequantized and SAVE_7B_TO_DRIVE and DRIVE_MODEL_DIR
        and not os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER))):
    try:
        print("Saving 4-bit copy to Drive (one-time, ~5 GB)...")
        model.save_pretrained(DRIVE_MODEL_DIR)
        tok.save_pretrained(DRIVE_MODEL_DIR)
        with open(os.path.join(DRIVE_MODEL_DIR, MARKER), "w") as f:
            f.write("ok\n")
        print("Saved to", DRIVE_MODEL_DIR)
    except Exception as e:
        print(f"Drive save skipped ({type(e).__name__}: {e}) — not fatal.")

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

print("trim_to_body ready")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

7B loaded 4-bit; 5.4 GB on GPU
trim_to_body ready


## 7-8. The RAG machinery (reused from Step 6, verbatim)

The next cells are the Step 6 retrieval stack unchanged — the MBPP corpus + TF-IDF index,
the ten rustc-verified idiom exemplars, and `build_rag_prompt` — so the exemplars the 7B
sees are byte-for-byte the ones the 1.5B saw. `ntokens`/`tok` now point at the 7B
tokenizer, which is what the budget check should use.

## 7. The RAG corpus — our MBPP translation pairs

The retrieval corpus is the same one Step 4 used at 350M: our validated MBPP
Python→Rust pairs. Each `rust_solution` is a complete, compiling function with its
`///` doc comment — exactly the shape of what we ask the model to write. As in
Step 4 we keep one exemplar per task (the shortest passing solution) to save
context tokens, and index with TF-IDF over character 3–5-grams.

One deliberate change from Step 4: **the query is the MultiPL-E prompt itself**
(doc comment + signature), and the document side is each pair's doc comment +
signature to match. Step 4 retrieved by Python-source similarity because its model
was a Python→Rust translator with the Python in hand; the vanilla completion task
has no Python at inference time, and retrieval must live with the same information
the model gets.

Source order: Drive `pairs_v3.jsonl` → `pairs_v2.jsonl` → the repo's
`notebooks/pairs.jsonl` (v1, 194 pairs — smaller, but keeps the notebook runnable
with zero Drive access). scikit-learn is preinstalled on Colab.

In [10]:
import json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_pairs():
    candidates = []
    if DRIVE_ROOT:
        candidates += [os.path.join(DRIVE_ROOT, c) for c in ("pairs_v3.jsonl", "pairs_v2.jsonl")]
    candidates += ["notebooks/pairs.jsonl", "pairs.jsonl", "../notebooks/pairs.jsonl"]
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                loaded = [json.loads(l) for l in f if l.strip()]
            print(f"Loaded {len(loaded)} pairs from {path}")
            if "pairs_v" not in os.path.basename(path):
                print("WARNING: this is the v1 fallback corpus (194 pairs). Fine for a dry run; "
                      "the real sweep should see pairs_v3 on Drive (643 pairs).")
            return loaded
    raise FileNotFoundError("no pairs file found (Drive pairs_v3/v2 or repo notebooks/pairs.jsonl)")

pairs = load_pairs()

# One exemplar per task: the SHORTEST solution (saves context tokens) — as in Step 4.
best = {}
for p in pairs:
    if p["task"] not in best or len(p["rust_solution"]) < len(best[p["task"]]["rust_solution"]):
        best[p["task"]] = p
corpus = list(best.values())

def doc_text(p):
    # What the retriever "sees" for an exemplar: doc comment + signature only,
    # to match the query (the MultiPL-E prompt = doc comment + signature).
    return p["rust_prompt"]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)
X = vec.fit_transform([doc_text(p) for p in corpus])
print(len(corpus), "unique tasks in the RAG index (from", len(pairs), "pairs)")

# Sanity check: an exemplar's own prompt must retrieve that exemplar first.
sims = cosine_similarity(vec.transform([doc_text(corpus[0])]), X)[0]
assert corpus[int(sims.argmax())]["task"] == corpus[0]["task"], "retriever failed its self-check"
print("retriever self-check OK")

Loaded 643 pairs from /content/drive/MyDrive/CodeGen_Group45/pairs_v3.jsonl
251 unique tasks in the RAG index (from 643 pairs)
retriever self-check OK


### 7b. The idiom exemplars — an error-analysis-guided corpus row

Step 5's error analysis says the compile errors concentrate in a few idioms:
indexing with `isize`, ordering floats, int/float mixing, helper functions that
were never defined, values moved instead of borrowed. The ten exemplars below
demonstrate exactly those idioms. They were written from the rustc error
*buckets*, not from any benchmark problem or its solution, so the corpus stays
disjoint from HumanEval.

**Design decision (measured, not guessed):** we first tried simply merging these
ten into the MBPP index — but TF-IDF then surfaces an idiom exemplar in the top-2
for only 4 of the 34 compile-error problems (the MBPP corpus is 20–60× larger and
usually closer in wording). Relevance-gated inclusion would make the row a no-op.
So the idiom row instead **always prepends the single most similar idiom
exemplar** (retrieved from the ten by the same TF-IDF machinery) on top of the K
retrieved MBPP exemplars: one targeted Rust lesson per prompt, guaranteed present.

**Honest caveat for the report:** the bucket frequencies were measured on the eval
set itself, so this row is development-set-informed in a way the plain MBPP sweep
is not. We keep both rows and say so.

In [11]:
IDIOM_EXEMPLARS = [
    {"task": "idiom_index_with_cast",
     "rust_prompt": "/// Return the element at position i of a list, where i is given as an isize.\nfn element_at(values: Vec<isize>, i: isize) -> isize {",
     "rust_solution": "/// Return the element at position i of a list, where i is given as an isize.\nfn element_at(values: Vec<isize>, i: isize) -> isize {\n    values[i as usize]\n}"},
    {"task": "idiom_max_of_floats",
     "rust_prompt": "/// Return the largest value in a non-empty list of floats.\nfn max_float(values: Vec<f64>) -> f64 {",
     "rust_solution": "/// Return the largest value in a non-empty list of floats.\nfn max_float(values: Vec<f64>) -> f64 {\n    values.iter().cloned().fold(f64::NEG_INFINITY, f64::max)\n}"},
    {"task": "idiom_sort_floats",
     "rust_prompt": "/// Return a copy of the list of floats sorted in increasing order.\nfn sort_floats(values: Vec<f64>) -> Vec<f64> {",
     "rust_solution": "/// Return a copy of the list of floats sorted in increasing order.\nfn sort_floats(values: Vec<f64>) -> Vec<f64> {\n    let mut sorted = values.clone();\n    sorted.sort_by(|a, b| a.partial_cmp(b).unwrap());\n    sorted\n}"},
    {"task": "idiom_nested_helper_fn",
     "rust_prompt": "/// Return true if the sum of the digits of n is a prime number.\nfn digit_sum_is_prime(n: isize) -> bool {",
     "rust_solution": "/// Return true if the sum of the digits of n is a prime number.\nfn digit_sum_is_prime(n: isize) -> bool {\n    fn is_prime(x: isize) -> bool {\n        if x < 2 {\n            return false;\n        }\n        for d in 2..=((x as f64).sqrt() as isize) {\n            if x % d == 0 {\n                return false;\n            }\n        }\n        true\n    }\n    let mut s = 0;\n    let mut m = n.abs();\n    while m > 0 {\n        s += m % 10;\n        m /= 10;\n    }\n    is_prime(s)\n}"},
    {"task": "idiom_int_float_mix",
     "rust_prompt": "/// Return the average of a non-empty list of integers as a float.\nfn average(values: Vec<isize>) -> f64 {",
     "rust_solution": "/// Return the average of a non-empty list of integers as a float.\nfn average(values: Vec<isize>) -> f64 {\n    let total: isize = values.iter().sum();\n    total as f64 / values.len() as f64\n}"},
    {"task": "idiom_widen_to_avoid_overflow",
     "rust_prompt": "/// Return the product of all elements, computed in i64 so it cannot overflow i32.\nfn product_wide(values: Vec<i32>) -> i64 {",
     "rust_solution": "/// Return the product of all elements, computed in i64 so it cannot overflow i32.\nfn product_wide(values: Vec<i32>) -> i64 {\n    values.iter().map(|&v| v as i64).product()\n}"},
    {"task": "idiom_string_compare",
     "rust_prompt": "/// Return true if two words are equal ignoring case.\nfn same_word(a: String, b: String) -> bool {",
     "rust_solution": "/// Return true if two words are equal ignoring case.\nfn same_word(a: String, b: String) -> bool {\n    a.to_lowercase() == b.to_lowercase()\n}"},
    {"task": "idiom_char_at_position",
     "rust_prompt": "/// Return the character at position k of a string (character, not byte, index).\nfn char_at(s: String, k: usize) -> char {",
     "rust_solution": "/// Return the character at position k of a string (character, not byte, index).\nfn char_at(s: String, k: usize) -> char {\n    s.chars().nth(k).unwrap_or(' ')\n}"},
    {"task": "idiom_iterate_by_reference",
     "rust_prompt": "/// Return the difference between the largest and smallest element,\nfn value_range(values: Vec<isize>) -> isize {",
     "rust_solution": "/// Return the difference between the largest and smallest element,\n/// iterating by reference so the vector is not moved.\nfn value_range(values: Vec<isize>) -> isize {\n    let max = values.iter().max().unwrap();\n    let min = values.iter().min().unwrap();\n    max - min\n}"},
    {"task": "idiom_enumerate_positions",
     "rust_prompt": "/// Return the positions at which a negative number appears in the list.\nfn negative_positions(values: Vec<isize>) -> Vec<usize> {",
     "rust_solution": "/// Return the positions at which a negative number appears in the list.\nfn negative_positions(values: Vec<isize>) -> Vec<usize> {\n    values.iter().enumerate()\n          .filter(|&(_, &v)| v < 0)\n          .map(|(i, _)| i)\n          .collect()\n}"},
]

# A separate tiny index over just the ten idioms: the idiom row always shows the
# ONE most similar idiom exemplar (see the markdown above for why it is not
# simply merged into the MBPP index).
vec_id = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)
X_id = vec_id.fit_transform([doc_text(p) for p in IDIOM_EXEMPLARS])

def retrieve(query_text, k):
    if k <= 0:
        return []
    sims = cosine_similarity(vec.transform([query_text]), X)[0]
    return [corpus[i] for i in sims.argsort()[::-1][:k]]

def retrieve_idiom(query_text):
    sims = cosine_similarity(vec_id.transform([query_text]), X_id)[0]
    return IDIOM_EXEMPLARS[int(sims.argmax())]

demo_q = "/// Return the largest element of a list of floats.\nfn max_elem(values: Vec<f64>) -> f64 {\n"
print("nearest idiom for a float task:", retrieve_idiom(demo_q)["task"])
print("nearest MBPP exemplars:        ", [d["task"] for d in retrieve(demo_q, 2)])

nearest idiom for a float task: idiom_max_of_floats
nearest MBPP exemplars:         ['mbpp_618_div_list', 'mbpp_251_insert_element']


## 8. RAG prompt assembly

Exemplars go ABOVE the problem prompt, as complete functions separated by blank
lines — pure Rust, the same completion format the baseline used. Nothing is added
to the problem prompt itself, so **K=0 is byte-identical to Step 5's prompt** —
asserted below for all 156 problems. That, plus greedy decoding, is what lets the
K=0 control reproduce 37.8% exactly.

A token budget caps the prompt: Qwen's 32K context is not the constraint, but long
prompts are slow on a T4. Same policy as Step 4 — nearest exemplar first, and an
exemplar that would overflow the budget is skipped, not truncated.

In [12]:
PROMPT_BUDGET = 4096 - 512   # prompt tokens; plenty for K=4 (median exemplar is small)

def ntokens(s):
    return len(tok(s)["input_ids"])

def build_rag_prompt(ex, k, use_idioms=False):
    exemplars = retrieve(ex["prompt"], k)                         # nearest first
    if use_idioms:
        exemplars = [retrieve_idiom(ex["prompt"])] + exemplars    # the one idiom lesson on top
    blocks, total = [], ntokens(ex["prompt"])
    for exemplar in exemplars:
        block = exemplar["rust_solution"].rstrip() + "\n\n"
        bt = ntokens(block)
        if total + bt > PROMPT_BUDGET:
            continue                     # would overflow the budget — skip, try next-nearest
        blocks.append(block)
        total += bt
    return "".join(blocks) + ex["prompt"], len(blocks)

# K=0 must be the baseline prompt, byte for byte — on every problem.
assert all(build_rag_prompt(ex, 0)[0] == ex["prompt"] for ex in ds)
print("K=0 identity check passed on all", len(ds), "problems")

demo_prompt, n_used = build_rag_prompt(ds[2], 2)
print(f"\ndemo: K=2 used {n_used} exemplars, {ntokens(demo_prompt)} prompt tokens")
print("-" * 60)
print(demo_prompt[:1200])

K=0 identity check passed on all 156 problems

demo: K=2 used 2 exemplars, 233 prompt tokens
------------------------------------------------------------
/// Write a function to convert the given decimal number to its binary equivalent, represented as a string with no leading zeros.
fn decimal_to_binary(n: isize) -> String {
    let mut binary_str = String::new();
    let mut n = n;
    loop {
        let rem = n % 2;
        binary_str.push_str(&rem.to_string());
        n /= 2;
        if n == 0 {
            break;
        }
    }
    binary_str.chars().rev().collect()
}

/// Write a rsthon function to find smallest number in a vector.
fn smallest_num(xs: Vec<isize>) -> isize {
    xs.iter().fold(isize::MAX, |a, &b| a.min(b))
}

/// Given a positive floating point number, it can be decomposed into
/// and integer part (largest integer smaller than given number) and decimals
/// (leftover part always smaller than 1).
/// Return the decimal part of the number.
/// >>> truncate_number(

## 9. The two 7B RAG sweeps the cascade needs

The cascade has three tiers: K=0 (vanilla), nearest-idiom+K=2, and K=4. The 7B already
produced K=0 in Step 8 (`step8_qwen7b_4bit.jsonl`), so we only generate the other two
here, writing to `step9_qwen7b_rag_*` files (no collision with the 1.5B's `step6_*`).
Smoke test first (house rule): 5 problems at K=2, throwaway file.

In [13]:
import time, json
from collections import Counter

EVAL_DIR = os.path.join(DRIVE_ROOT, "eval") if DRIVE_ROOT else "."

def rag_completion(prompt_text, max_new_tokens=512):
    inputs = tok(prompt_text, return_tensors="pt").to(GEN_DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

def run_k(k, use_idioms=False, limit=None, tag=""):
    path = os.path.join(EVAL_DIR, f"step9_qwen7b_rag_k{k}{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                done[rec["name"]] = rec["status"]
    data = ds if limit is None else ds[:limit]
    todo = [ex for ex in data if ex["name"] not in done]
    print(f"7B K={k}{tag}: {len(done)} done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            prompt_text, n_used = build_rag_prompt(ex, k, use_idioms)
            body = rag_completion(prompt_text)
            status = evaluate_one(ex["prompt"], body, ex["tests"])
            out.write(json.dumps({"name": ex["name"], "k": k, "n_examples": n_used,
                                  "status": status, "body": body}) + "\n")
            out.flush()
            done[ex["name"]] = status
            print(f"[{len(done):3d}/{len(data)}] {ex['name'][:40]:40s} {status:14s} ({time.time()-t0:5.0f}s)")
    counts = Counter(done.values())
    print(f"7B K={k}{tag}: pass {100*counts['pass']/len(done):.1f}%  {dict(counts)}")
    return counts

def load_run(fname):
    path = os.path.join(EVAL_DIR, fname)
    if not os.path.exists(path):
        return None
    d = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            d[r["name"]] = {"status": r["status"], "body": r["body"]}
    return d if len(d) == len(ds) else None

# --- smoke test: 5 problems at K=2, throwaway file (house rule) ---
smoke = run_k(2, limit=5, tag="_smoke")
os.remove(os.path.join(EVAL_DIR, "step9_qwen7b_rag_k2_smoke.jsonl"))
assert smoke["compile_error"] < 5, "every smoke problem failed to compile — inspect a prompt/body first"
print("\nsmoke OK — the two sweeps below are safe to launch")

7B K=2_smoke: 0 done, 5 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step9_qwen7b_rag_k2_smoke.jsonl
[  1/5] HumanEval_0_has_close_elements           pass           (    9s)
[  2/5] HumanEval_1_separate_paren_groups        pass           (   18s)
[  3/5] HumanEval_2_truncate_number              pass           (   20s)
[  4/5] HumanEval_3_below_zero                   pass           (   24s)
[  5/5] HumanEval_4_mean_absolute_deviation      compile_error  (   30s)
7B K=2_smoke: pass 80.0%  {'pass': 4, 'compile_error': 1}

smoke OK — the two sweeps below are safe to launch


In [14]:
# The two tiers the cascade still needs (K=0 comes from Step 8). ~20-25 min each on a T4.
run_k(2, use_idioms=True, tag="_idiom")   # -> step9_qwen7b_rag_k2_idiom.jsonl
run_k(4)                                  # -> step9_qwen7b_rag_k4.jsonl

7B K=2_idiom: 0 done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step9_qwen7b_rag_k2_idiom.jsonl
[  1/156] HumanEval_0_has_close_elements           pass           (    4s)
[  2/156] HumanEval_1_separate_paren_groups        pass           (   15s)
[  3/156] HumanEval_2_truncate_number              pass           (   16s)
[  4/156] HumanEval_3_below_zero                   pass           (   20s)
[  5/156] HumanEval_4_mean_absolute_deviation      compile_error  (   25s)
[  6/156] HumanEval_5_intersperse                  pass           (   29s)
[  7/156] HumanEval_6_parse_nested_parens          pass           (   39s)
[  8/156] HumanEval_7_filter_by_substring          pass           (   42s)
[  9/156] HumanEval_8_sum_product                  pass           (   44s)
[ 10/156] HumanEval_9_rolling_max                  pass           (   49s)
[ 11/156] HumanEval_10_make_palindrome             run_fail       (   57s)
[ 12/156] HumanEval_11_string_xor                  pass         

Counter({'pass': 87, 'run_fail': 45, 'compile_error': 23, 'run_timeout': 1})

## 10. The 7B compile-guided cascade

K=0 (Step 8 vanilla) -> idiom+k2 -> k4, taking the first body that compiles. This is the
identical policy that took the 1.5B from 37.8% to 44.9%; here we measure what it does to
the 7B's 58.3%.

In [15]:
k0  = load_run("step8_qwen7b_4bit.jsonl")        # 7B vanilla == K=0
idi = load_run("step9_qwen7b_rag_k2_idiom.jsonl")
k4  = load_run("step9_qwen7b_rag_k4.jsonl")

assert k0 is not None, "step8_qwen7b_4bit.jsonl (156 rows) missing — run Step 8 first."
assert idi is not None and k4 is not None, "run the two sweeps in the cell above first."

def cascade_pick(name):
    for src in (k0, idi, k4):            # first body that compiles wins
        if src[name]["status"] != "compile_error":
            return {"status": src[name]["status"], "body": src[name]["body"]}
    return {"status": k4[name]["status"], "body": k4[name]["body"]}

cascade_start = {ex["name"]: cascade_pick(ex["name"]) for ex in ds}
cc = Counter(v["status"] for v in cascade_start.values())
n = len(ds)
print("Same harness, same 156 problems")
print(f"  Qwen-1.5B cascade+repair (best small) :  72/156 = 46.2%")
print(f"  Qwen-7B-4bit vanilla (Step 8, = K=0)  :  91/156 = 58.3%")
print(f"  Qwen-7B-4bit compile-guided cascade   : {cc['pass']:3d}/156 = {100*cc['pass']/n:.1f}%")
print(f"    full mix: {dict(cc)}")

Same harness, same 156 problems
  Qwen-1.5B cascade+repair (best small) :  72/156 = 46.2%
  Qwen-7B-4bit vanilla (Step 8, = K=0)  :  91/156 = 58.3%
  Qwen-7B-4bit compile-guided cascade   :  97/156 = 62.2%
    full mix: {'pass': 97, 'run_fail': 47, 'compile_error': 10, 'run_timeout': 2}


## 11. Compiler-feedback repair on top of the 7B cascade

Whatever still fails to compile after the cascade goes back to the 7B with the exact rustc
error (Step 7 base-model repair: the broken function and the compiler's complaint as
comments, then the signature again for completion), up to two rounds. Only compile errors
enter the loop, so passes and run-fails cannot regress — the number can only rise.

In [16]:
import subprocess, tempfile, os

def compile_message(prompt, completion, tests, compile_timeout=60):
    """Compile prompt+completion+tests; return '' if it builds, else rustc's stderr.
    Sibling of evaluate_one (kept byte-identical); only this helper reads the message."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "error: compilation timed out"
        return "" if c.returncode == 0 else c.stderr

def _as_comment(text):
    return "\n".join("// " + line for line in text.splitlines())

MAX_ERR_CHARS = 1200

def build_repair_prompt(ex, broken_body, error_text):
    broken_fn = ex["prompt"] + broken_body
    err = error_text.strip()
    if len(err) > MAX_ERR_CHARS:
        err = err[:MAX_ERR_CHARS] + "\n... (truncated)"
    header = (
        "// This Rust function did not compile.\n"
        f"{_as_comment(broken_fn)}\n"
        "//\n"
        "// The Rust compiler reported:\n"
        f"{_as_comment(err)}\n"
        "//\n"
        "// Corrected version that compiles:\n"
    )
    return header + ex["prompt"]

def repair_completion(prompt_text, max_new_tokens=512):
    inputs = tok(prompt_text, return_tensors="pt").to(GEN_DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

REPAIR_ROUNDS = 2

def run_repair(start, max_rounds=REPAIR_ROUNDS, names=None, tag=""):
    """Only compile errors enter the loop; everything else copies through. Streams+resumes."""
    path = os.path.join(EVAL_DIR, f"step9_qwen7b_repair_r{max_rounds}{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                done[r["name"]] = r
    targets = [ex for ex in ds if (names is None or ex["name"] in names)]
    todo = [ex for ex in targets if ex["name"] not in done]
    print(f"7B repair r{max_rounds}{tag}: {len(done)} done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            name = ex["name"]
            status = start[name]["status"]
            body = start[name]["body"]
            trail = [status]
            rounds = 0
            while status == "compile_error" and rounds < max_rounds:
                err = compile_message(ex["prompt"], body, ex["tests"])
                body = repair_completion(build_repair_prompt(ex, body, err))
                status = evaluate_one(ex["prompt"], body, ex["tests"])
                trail.append(status)
                rounds += 1
            rec = {"name": name, "baseline": trail[0], "status": status,
                   "rounds": rounds, "trail": trail, "body": body}
            out.write(json.dumps(rec) + "\n")
            out.flush()
            done[name] = rec
            if trail[0] == "compile_error":
                print(f"[{len(done):3d}/{len(targets)}] {name[:36]:36s} "
                      f"{trail[0]:13s} -> {status:13s} ({rounds}r, {time.time()-t0:4.0f}s)")
    return done

print("7B repair pipeline ready")

7B repair pipeline ready


In [17]:
composed = run_repair(cascade_start, max_rounds=REPAIR_ROUNDS)
cp = Counter(r["status"] for r in composed.values())
n = len(ds)
print("Same harness, same 156 problems")
print(f"  Qwen-1.5B cascade+repair (best small)      :  72/156 = 46.2%")
print(f"  Qwen-7B-4bit vanilla (Step 8)              :  91/156 = 58.3%")
print(f"  Qwen-7B-4bit cascade                       : {cc['pass']:3d}/156 = {100*cc['pass']/n:.1f}%")
print(f"  Qwen-7B-4bit cascade + repair (this step)  : {cp['pass']:3d}/156 = {100*cp['pass']/n:.1f}%")
print(f"    remaining compile errors: {cp['compile_error']}   full mix: {dict(cp)}")

ce = [r for r in composed.values() if r["baseline"] == "compile_error"]
fixed = [r for r in ce if r["status"] == "pass"]
print(f"\nof the {len(ce)} compile errors left after the cascade, repair recovered {len(fixed)} to pass:")
print("   ", [r["name"] for r in fixed])

7B repair r2: 0 done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step9_qwen7b_repair_r2.jsonl
[ 20/156] HumanEval_19_sort_numbers            compile_error -> compile_error (2r,   34s)
[ 91/156] HumanEval_94_skjkasdkd               compile_error -> compile_error (2r,   52s)
[119/156] HumanEval_123_get_odd_collatz        compile_error -> compile_error (2r,   65s)
[122/156] HumanEval_127_intersection           compile_error -> compile_error (2r,   81s)
[135/156] HumanEval_141_file_name_check        compile_error -> run_fail      (1r,  108s)
[137/156] HumanEval_143_words_in_sentence      compile_error -> compile_error (2r,  119s)
[139/156] HumanEval_145_order_by_points        compile_error -> compile_error (2r,  131s)
[141/156] HumanEval_147_get_max_triples        compile_error -> pass          (1r,  142s)
[143/156] HumanEval_150_x_or_y                 compile_error -> compile_error (2r,  147s)
[155/156] HumanEval_162_string_to_md5          compile_error -> compile_error (2r,

## What this step adds

- The **complete comparison**: our compile-guided cascade + compiler-feedback repair
  applied at both scales. If the 7B cascade/repair clears 58.3%, it shows the method is
  **complementary to scale**, not something a bigger model makes redundant — the strongest
  version of the report's thesis.
- If the lift is small, that is itself the honest finding: at 7B the compile bucket is
  already thin (the 7B had 23 compile errors vs the 1.5B's 34), so retrieval and repair
  have less to grab — scale absorbs the very failures our pipeline targets.

Either way, the top line of the whole project is now measured. Remaining work is assembly:
fold every number into `CP3_COMPARISON.md` and finalize the demo for deployment.